---

<h3><center>E178/ME276DS - Statistics and Data Science for Engineers</center></h3>

<h1><center>Final Project Data Cleaning<br></center></h1>

---

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt



In [2]:
data_PressureLevel = pd.read_csv('Data/Pressure Level Data.csv')
data_CloudandRain = pd.read_csv('Data/Cloud Coverage and Precipitation.csv')

In [12]:
print(data_PressureLevel.columns)
print(data_PressureLevel.head(5))
print('-'*90)
print(data_CloudandRain.columns)
print(data_CloudandRain.head(5))
print('-'*90)



Index(['valid_time', 'pressureLevel', 'z', 'u', 'v', 'r', 't', 'latitude',
       'longitude'],
      dtype='object')
  valid_time  pressureLevel           z         u         v          r  \
0 1995-03-01           1000   1382.3281  1.648590 -0.128830  55.401173   
1 1995-03-01            925   7822.1400  1.654526 -0.131912  55.169790   
2 1995-03-01            850  14707.0625  1.616882 -0.123596  55.593460   
3 1995-03-01            700  30132.7770  1.595703 -0.106598  54.917370   
4 1995-03-01            600  42007.5860  1.717270  0.065689  52.694664   

           t  latitude  longitude  
0  292.19580        28         87  
1  287.90283        28         87  
2  283.36716        28         87  
3  273.04413        28         87  
4  265.12560        28         87  
------------------------------------------------------------------------------------------
Index(['valid_time', 'tcc', 'tp', 'latitude', 'longitude'], dtype='object')
           valid_time       tcc        tp  latitude  l

In [13]:
# Care about 300 and 400 Pressure Level
PL_mask = data_PressureLevel['pressureLevel'].between(290,410)
data_PressureLevel = data_PressureLevel.loc[PL_mask,:].reset_index(drop = True)
data_PressureLevel['r'] = data_PressureLevel['r']/ 100

In [14]:
data_PressureLevel['valid_time'] = pd.to_datetime(data_PressureLevel['valid_time'], errors = 'coerce',format ='mixed')
data_CloudandRain['valid_time'] = pd.to_datetime(data_CloudandRain['valid_time'], errors = 'coerce',format='mixed')

In [15]:
# Limtied to every six hours by Pressure Level Data, match hourly data to every 6 hours
time_mask = data_CloudandRain['valid_time'].dt.hour.isin([0,6,12,18])
data_CloudandRain = data_CloudandRain.loc[time_mask,:].reset_index(drop = True)

In [ ]:
# print(data_PressureLevel.shape)
# print(data_PressureLevel.info)
# print('-'*50)
# print(data_CloudandRain.shape)
# print(data_CloudandRain.info)

(91560, 9)
<bound method DataFrame.info of                valid_time  pressureLevel          z          u         v  \
0     1995-03-01 00:00:00            400  71847.530  29.574905 -7.181381   
1     1995-03-01 00:00:00            300  92209.250  65.017290 -1.983643   
2     1995-03-01 06:00:00            400  72200.710  28.495514 -8.949142   
3     1995-03-01 06:00:00            300  92606.125  63.349182  2.758011   
4     1995-03-01 12:00:00            400  72245.750  27.578110 -8.786606   
...                   ...            ...        ...        ...       ...   
91555 2026-06-30 06:00:00            300  95974.875  -0.640320 -1.735642   
91556 2026-06-30 12:00:00            400  74713.800  -4.884201 -4.708374   
91557 2026-06-30 12:00:00            300  95971.500  -3.456314 -6.426987   
91558 2026-06-30 18:00:00            400  74759.030  -2.274124 -1.431335   
91559 2026-06-30 18:00:00            300  95896.500  -7.163544 -4.599396   

              r          t  latitude  longit

In [24]:
# 1. Pivot pressure levels (300 and 400) so each timestamp is 1 row
vars_to_pivot = ['z', 'u', 'v', 'r', 't']

df_pressure_wide = data_PressureLevel.pivot(
    index=['valid_time', 'latitude', 'longitude'],
    columns='pressureLevel',
    values=vars_to_pivot
)

# 2. Flatten MultiIndex columns into clean names (e.g., 'z_300', 'z_400', 't_300', 't_400')
df_pressure_wide.columns = [f"{var}_{int(lvl)}" for var, lvl in df_pressure_wide.columns]
df_pressure_wide = df_pressure_wide.reset_index()

# 3. Splice the two DataFrames together on valid_time, latitude, and longitude
merged_data = pd.merge(
    data_CloudandRain,
    df_pressure_wide,
    on=['valid_time', 'latitude', 'longitude'],
    how='inner'
)

print(merged_data.shape)
merged_data.head(30)

(45780, 15)


,valid_time,tcc,tp,latitude,longitude,z_300,z_400,u_300,u_400,v_300,v_400,r_300,r_400,t_300,t_400
0,1995-03-01 00:00:00,0.064667,0.000034,28,87,92209.250,71847.530,65.017290,29.574905,-1.983643,-7.181381,0.049329,0.041893,242.74450,249.90230
1,1995-03-01 06:00:00,0.079956,0.000025,28,87,92606.125,72200.710,63.349182,28.495514,2.758011,-8.949142,0.055107,0.038382,242.77657,250.55923
2,1995-03-01 12:00:00,0.053314,0.000003,28,87,92566.190,72245.750,61.283950,27.578110,0.848953,-8.786606,0.063069,0.067576,242.20369,249.99225
3,1995-03-01 18:00:00,0.009338,0.000000,28,87,92966.440,72589.484,59.922836,27.462875,8.558640,-8.623581,0.060122,0.157725,243.27147,249.72462
4,1995-03-02 00:00:00,0.041992,0.000003,28,87,92880.875,72472.310,55.527267,35.447480,10.592300,-2.682632,0.049564,0.054815,242.28711,250.93219
5,1995-03-02 06:00:00,0.063965,0.000021,28,87,93201.625,72732.170,55.049164,40.046585,10.150406,-2.018753,0.052975,0.043462,242.41414,252.16872
6,1995-03-02 12:00:00,0.258728,0.000021,28,87,92948.940,72527.580,57.786910,41.035280,12.471039,-2.952454,0.044513,0.058761,242.71840,251.52496
7,1995-03-02 18:00:00,0.177307,0.000029,28,87,92944.690,72547.695,60.926240,39.945390,14.707184,0.128159,0.041528,0.045742,243.35085,250.48760
8,1995-03-03 00:00:00,0.092651,0.000015,28,87,92666.310,72404.240,60.108870,38.057755,13.778305,-5.499176,0.040959,0.093553,242.04222,249.82149
9,1995-03-03 06:00:00,0.102631,0.000030,28,87,92741.440,72633.740,53.502120,37.740950,6.251526,-4.475586,0.109552,0.112957,239.19885,250.12581
